# Resumo Comparativo — Experimentos BRKGA

Consolida os CSVs de resultados por instância e gera a tabela comparativa global.

In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'utils' else cwd
RESULTADOS_DIR = ROOT / 'resultados' / 'experimentos_brkga'

print(f'ROOT:           {ROOT}')
print(f'RESULTADOS_DIR: {RESULTADOS_DIR}')

ROOT:           /mnt/Bau/Estudo/UFMA/LACMOR/FlowShopScheduling
RESULTADOS_DIR: /mnt/Bau/Estudo/UFMA/LACMOR/FlowShopScheduling/resultados/experimentos_brkga


In [3]:
# 1. Lista com os CSVs dos experimentos que você quer comparar
ARQUIVOS_CSV = [
    RESULTADOS_DIR / 'brkga_scs_dmn' / 'results_brkga_scs_dmn_60_makespan_taillard.csv',
    RESULTADOS_DIR / 'brkga_scs_img' / 'results_brkga_scs_img_60_makespan_taillard.csv',
    RESULTADOS_DIR / 'brkga_scs_vm' / 'results_brkga_scs_vm_60_makespan_taillard.csv',
    RESULTADOS_DIR / 'brkga_scs_vr' / 'results_brkga_scs_vr_60_makespan_taillard.csv',
    RESULTADOS_DIR / 'brkga_rsc' / 'results_brkga_rsc_60_makespan_taillard.csv',
]

# Caminho onde será salvo o novo CSV consolidado (médias por instância x método)
CSV_RESUMO_PATH = RESULTADOS_DIR / 'resumo_comparativo_por_instancia.csv'
CSV_RESUMO_GLOBAL_PATH = RESULTADOS_DIR / 'resumo_comparativo_global.csv'

# 2. Carrega e junta todos os CSVs da lista que existirem
dfs = []
for caminho in ARQUIVOS_CSV:
    if caminho.exists():
        dfs.append(pd.read_csv(caminho))
    else:
        print(f'[AVISO] Arquivo não encontrado (ignorado): {caminho.name}')

if not dfs:
    raise FileNotFoundError('Nenhum dos arquivos CSV informados foi encontrado!')

df_total = pd.concat(dfs, ignore_index=True)
print(f'Total de execuções carregadas ({len(dfs)} arquivos): {len(df_total)} linhas\n')

# 3. Agrupa por Instância + Método (calculando a média das seeds)
agg_instancia = df_total.groupby(['instancia', 'n_jobs', 'n_machines', 'metodo'], as_index=False).agg(
    makespan_medio=('makespan', 'mean'),
    tardiness_medio=('tardiness', 'mean'),
    desvio=('makespan', 'std'),
    melhor=('makespan', 'min'),
    pior=('makespan', 'max'),
    tempo_medio=('tempo_s', 'mean'),
    bks=('bks', 'first'),
    rpd_medio=('rpd', 'mean'),
    lower_bound=('lower_bound', 'first'),
)

# Ordena por tamanho da instância (jobs, máquinas), nome da instância e método
agg_instancia = agg_instancia.sort_values(['n_jobs', 'n_machines', 'instancia', 'metodo']).reset_index(drop=True)

# Arredondamento das colunas numéricas
agg_instancia['makespan_medio'] = agg_instancia['makespan_medio'].round(1)
agg_instancia['tardiness_medio'] = agg_instancia['tardiness_medio'].round(1)
agg_instancia['desvio'] = agg_instancia['desvio'].round(1)
agg_instancia['tempo_medio'] = agg_instancia['tempo_medio'].round(3)
agg_instancia['rpd_medio'] = agg_instancia['rpd_medio'].round(2)

# Salva o CSV detalhado por instância
agg_instancia.to_csv(CSV_RESUMO_PATH, index=False)

print('=' * 130)
print('TABELA COMPARATIVA — Média das Seeds por Instância x Método')
print('=' * 130)
print(agg_instancia.to_string(index=False))
print()

# 4. Resumo Global (agregando todas as instâncias por Método)
global_agg = df_total.groupby('metodo', as_index=False).agg(
    makespan_medio=('makespan', 'mean'),
    tardiness_medio=('tardiness', 'mean'),
    tempo_medio=('tempo_s', 'mean'),
    rpd_medio=('rpd', 'mean'),
)

global_agg['makespan_medio'] = global_agg['makespan_medio'].round(1)
global_agg['tardiness_medio'] = global_agg['tardiness_medio'].round(1)
global_agg['tempo_medio'] = global_agg['tempo_medio'].round(3)
global_agg['rpd_medio'] = global_agg['rpd_medio'].round(2)

# Salva também o resumo global em CSV
global_agg.to_csv(CSV_RESUMO_GLOBAL_PATH, index=False)

print('-' * 80)
print('RESUMO GLOBAL — Média Geral por Método')
print('-' * 80)
print(global_agg.to_string(index=False))
print()
print(f'CSV detalhado (por instância) salvo em: {CSV_RESUMO_PATH}')
print(f'CSV resumo global salvo em:             {CSV_RESUMO_GLOBAL_PATH}')

Total de execuções carregadas (5 arquivos): 3000 linhas

TABELA COMPARATIVA — Média das Seeds por Instância x Método
instancia  n_jobs  n_machines        metodo  makespan_medio  tardiness_medio  desvio  melhor    pior  tempo_medio   bks  rpd_medio  lower_bound
ta001.txt      20           5     brkga_rsc          1278.0          14938.0     0.0  1278.0  1278.0        3.020  1278       0.00       1249.0
ta001.txt      20           5 brkga_scs_dmn          1298.2          15735.8     2.7  1297.0  1303.0        3.039  1278       1.58       1249.0
ta001.txt      20           5 brkga_scs_img          1297.0          15969.0     0.0  1297.0  1297.0        3.030  1278       1.49       1249.0
ta001.txt      20           5  brkga_scs_vm          1297.0          16049.8     0.0  1297.0  1297.0        3.022  1278       1.49       1249.0
ta001.txt      20           5  brkga_scs_vr          1302.0          15718.6    11.2  1297.0  1322.0        3.053  1278       1.88       1249.0
ta002.txt      20  